In [1]:
import sys
import os
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Add parent directory to path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Ensure spacy model is downloaded (run this in terminal if needed: python -m spacy download en_core_web_sm)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading Spacy model...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print(f"System ready. GPU Available: {torch.cuda.is_available()}")

/opt/conda/envs/sentinel-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 11.8/12.8 MB 15.1 MB/s  0:00:010m


  Resuming download https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (11.8 MB/12.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 3.8 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
System ready. GPU Available: False


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# Use float16 for speed if on GPU
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_name = "google/flan-t5-base"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, 
    torch_dtype=torch_dtype
).to(device)

print("Model loaded.")

Loading google/flan-t5-base...
Model loaded.


In [3]:
def needs_decontextualization(text: str) -> bool:
    """
    Paper-Aligned Logic:
    1. Check for pronouns (He, She, It, They).
    2. Check for demonstratives (This, That, These).
    3. Check for starting conjunctions (But, And, However) - implies connection to prev sentence.
    """
    doc = nlp(text)
    
    # 1. Pronouns & Demonstratives
    target_words = {"this", "that", "these", "those", "he", "she", "it", "they", "his", "her", "its", "their"}
    for token in doc:
        if token.lower_ in target_words:
            return True
        if token.pos_ == "PRON":
            return True
            
    # 2. Starting Conjunctions (Context dependency)
    # If the sentence starts with 'But', 'However', 'Therefore', it relies on previous text.
    if doc[0].text.lower() in ["but", "however", "therefore", "and", "so"]:
        return True
            
    return False

# Test
tests = [
    "The economy is stable.", # False
    "However, it might crash soon.", # True (Conjunction + Pronoun)
    "This is unacceptable.", # True (Demonstrative)
    "Biden signed the bill." # False
]
for t in tests:
    print(f"'{t}' -> Needs Decontext? {needs_decontextualization(t)}")

'The economy is stable.' -> Needs Decontext? False
'However, it might crash soon.' -> Needs Decontext? True
'This is unacceptable.' -> Needs Decontext? True
'Biden signed the bill.' -> Needs Decontext? False


In [4]:
def resolve_coreference(sentence: str, full_context: str) -> str:
    """
    Rewrites the sentence to be standalone.
    """
    # 1. Filter
    if not needs_decontextualization(sentence):
        return sentence

    # 2. Prompt (Aligned with Paper's "Rewrite" objective)
    # We pass the full context but the model attends to what's needed.
    # In a real app, 'full_context' should be just the prev 1-3 sentences.
    prompt = (
        f"Rewrite this sentence to be fully explicit and standalone. "
        f"Resolve any pronouns or references using the context.\n\n"
        f"Context: {full_context}\n\n"
        f"Sentence: {sentence}\n\n"
        f"Output:"
    )
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids, 
            max_length=128, 
            num_beams=4, 
            early_stopping=True
        )
    
    resolved_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 3. Sanity Check
    # If output is broken or empty, return original
    if not resolved_text or len(resolved_text) < 10:
        return sentence
        
    return resolved_text



In [5]:
# Simulating a snippet where context is crucial
article_snippet = """
The new health bill was debated in the Senate yesterday.
Critics argued it would increase the deficit by billions.
However, supporters claimed it was necessary for reform.
"""

target_claims = [
    "Critics argued it would increase the deficit by billions.",
    "However, supporters claimed it was necessary for reform."
]

print(f"{'ORIGINAL':<60} | {'RESOLVED'}")
print("-" * 100)

for claim in target_claims:
    resolved = resolve_coreference(claim, article_snippet)
    print(f"{claim:<60} | \033[92m{resolved}\033[0m")

ORIGINAL                                                     | RESOLVED
----------------------------------------------------------------------------------------------------
Critics argued it would increase the deficit by billions.    | Critics argued it would increase the deficit by billions.
However, supporters claimed it was necessary for reform.     | The new health bill was debated in the Senate yesterday. Critics argued it would increase the deficit by billions. However, supporters claimed it was necessary for reform.


In [6]:
# scripts/06_decontext.ipynb

import sys
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Setup
MODEL_NAME = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# Load Spacy for splitting (simulating the Preprocessor)
try:
    nlp = spacy.load("en_core_web_sm")
except:
    print("Please download spacy model: python -m spacy download en_core_web_sm")
    sys.exit()

# 2. Define the Sliding Window Logic
def decontextualize_article(raw_text, window_size=3):
    """
    Processes a full text by using a sliding window of previous sentences as context.
    """
    # Step A: Split into sentences
    doc = nlp(raw_text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    
    results = []
    
    print(f"Processing {len(sentences)} sentences with Window Size {window_size}...\n")
    
    # Step B: Iterate
    for i, current_sentence in enumerate(sentences):
        # 1. Build Context
        # Grab the previous 'window_size' sentences
        start_index = max(0, i - window_size)
        previous_sentences = sentences[start_index:i]
        context_block = " ".join(previous_sentences)
        
        # If it's the very first sentence, the context is empty (or we could use title)
        if not context_block:
            context_block = "Start of article."

        # 2. Prepare Prompt
        input_text = (
            f"Context: {context_block}\n\n"
            f"Sentence: {current_sentence}\n\n"
            f"Rewrite the sentence to be self-contained by replacing pronouns (he, she, it, they) "
            f"and generic terms with specific names from the context. "
            f"If no change is needed, output the original sentence.\n\n"
            f"Rewritten:"
        )
        
        # 3. Inference
        inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)
        
        outputs = model.generate(
            inputs.input_ids, 
            max_length=128, 
            num_beams=5, 
            early_stopping=True
        )
        
        rewritten_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # 4. Store Result
        # Only print if it actually changed something important
        if rewritten_text.lower() != current_sentence.lower():
            change_status = "MODIFIED"
        else:
            change_status = "SAME"
            
        results.append({
            "Original": current_sentence,
            "Context Used": context_block[:100] + "..." if len(context_block) > 100 else context_block,
            "Rewritten": rewritten_text,
            "Status": change_status
        })

    return results

# 3. The "Real Article" Simulation
full_article_text = """
Elon Musk took the stage at the Starbase facility in Texas today. 
He announced that the next Starship launch is scheduled for March. 
The CEO emphasized that safety is the primary concern for the team.
However, critics argue that he is rushing the timeline to please investors.
They believe the vehicle is not yet ready for orbital flight.
It has faced several delays in the past year.
"""

# 4. Run it
processed_data = decontextualize_article(full_article_text, window_size=2)

# 5. Display
print("-" * 80)
print(f"{'STATUS':<10} | {'ORIGINAL':<50} | {'REWRITTEN'}")
print("-" * 80)

for row in processed_data:
    if row['Status'] == "MODIFIED":
        print(f"\033[92m{row['Status']:<10}\033[0m | {row['Original'][:50]:<50} | \033[92m{row['Rewritten']}\033[0m")
    else:
        print(f"{row['Status']:<10} | {row['Original'][:50]:<50} | {row['Rewritten']}")

Loading google/flan-t5-base on cpu...
Processing 6 sentences with Window Size 2...

--------------------------------------------------------------------------------
STATUS     | ORIGINAL                                           | REWRITTEN
--------------------------------------------------------------------------------
SAME       | Elon Musk took the stage at the Starbase facility  | Elon Musk took the stage at the Starbase facility in Texas today.
MODIFIED   | He announced that the next Starship launch is sche | Elon Musk took the stage at the Starbase facility in Texas today.
MODIFIED   | The CEO emphasized that safety is the primary conc | Elon Musk took the stage at the Starbase facility in Texas today. He announced the next Starship launch is scheduled for March.
MODIFIED   | However, critics argue that he is rushing the time | He announced that the next Starship launch is scheduled for March. Critics argue that he is rushing the timeline to please investors.
MODIFIED   | They be